# Data Mining - Project 

##### Work done by
- Roberto Rolão de Castro 107133
- Tiago Caridade Gomes 108307

## Tracking the Evolution of E-commerce Websites in Portugal 
**Description**: Study how e-commerce websites in Portugal, such as those related to retail (e.g., Worten, Fnac) have changed in terms of offerings and products availability.  
**Data** : Gather versions of popular e-commerce sites over time from Arquivo.pt.

In [31]:
import requests
import pandas as pd

def read_list_from_file(filename):
    """Reads a list of items from a file, one item per line."""
    with open(filename, 'r') as file:
        return [line.strip() for line in file if line.strip()]

def fetch_all_results(websites, keywords):
    url = "https://arquivo.pt/textsearch"
    all_data = []
    seen_urls = set()  # To track the URLs that have already been added

    for website in websites:
        for keyword in keywords:
            print(f"Starting search for '{keyword}' on {website}")
            base_params = {
                "from": "20000101000000",
                "to": "20240101000000",
                "maxItems": 500,
                "q": keyword,
                "siteSearch": website,
                "dedupValue": 0,
                "dedupField": "url",
                "offset": 0
            }

            # Fetch data for each combination of website and keyword
            while True:
                response = requests.get(url, params=base_params)
                if response.status_code != 200:
                    print(f"Failed to fetch data at offset {base_params['offset']}. Status code: {response.status_code}")
                    break

                data = response.json()
                if 'response_items' in data and data['response_items']:
                    current_data = pd.DataFrame(data['response_items'])

                    # Filter out duplicates based on originalURL
                    new_data = current_data[~current_data['originalURL'].isin(seen_urls)]
                    if not new_data.empty:
                        all_data.append(new_data)
                        # Add the new originalURLs to the seen set
                        seen_urls.update(new_data['originalURL'])
                        print(f"Added {len(new_data)} new items for '{keyword}' on {website} at offset {base_params['offset']}")
                    else:
                        print(f"No new items found for '{keyword}' on {website} at offset {base_params['offset']}")

                    # Check if fewer results were returned than maxItems, indicating the last page
                    if len(current_data) < base_params['maxItems']:
                        print("Reached the last page of data.")
                        break

                    base_params['offset'] += base_params['maxItems']
                else:
                    print("No more data available or reached API limit.")
                    break

            # Save the results for the current website and keyword
            if all_data:
                final_data = pd.concat(all_data, ignore_index=True)
                # Drop duplicates from the final concatenated DataFrame based on originalURL
                final_data = final_data.drop_duplicates(subset=['originalURL'])
                output_filename = f"./Results/{website.replace('https://', '').replace('.', '_')}_{keyword}.csv"
                final_data.to_csv(output_filename, index=False)
                print(f"Data saved to {output_filename}")
                all_data.clear()  # Clear the list for the next combination
            else:
                print(f"No data found for '{keyword}' on {website}")

def main():
    # Read websites and keywords from files
    websites = read_list_from_file('websites.txt')
    keywords = read_list_from_file('categories.txt')

    # Fetch all results
    fetch_all_results(websites, keywords)

if __name__ == "__main__":
    main()


Starting search for 'Telemóvel' on https://www.worten.pt
Added 500 new items for 'Telemóvel' on https://www.worten.pt at offset 0
Added 472 new items for 'Telemóvel' on https://www.worten.pt at offset 500
Added 257 new items for 'Telemóvel' on https://www.worten.pt at offset 1000
Added 291 new items for 'Telemóvel' on https://www.worten.pt at offset 1500
No more data available or reached API limit.
Data saved to ./Results/www_worten_pt_Telemóvel.csv
Starting search for 'Smartphone' on https://www.worten.pt
Added 480 new items for 'Smartphone' on https://www.worten.pt at offset 0
Added 494 new items for 'Smartphone' on https://www.worten.pt at offset 500
Added 421 new items for 'Smartphone' on https://www.worten.pt at offset 1000
Added 380 new items for 'Smartphone' on https://www.worten.pt at offset 1500
No more data available or reached API limit.
Data saved to ./Results/www_worten_pt_Smartphone.csv
Starting search for 'Portátil' on https://www.worten.pt
Added 496 new items for 'Portá

WEB Scrapping